# Data Loading and Initial Exploration

In this section, the datasets used throughout the project are loaded
into the notebook. Two datasets are used:

1. `matches.csv` — contains match-level information such as teams,
   goals, attendance, and expected goals (xG).

2. `seasonstats.csv` — contains aggregated season statistics for
   each Premier League team, including attacking and defensive metrics.

The purpose of this step is to inspect the available variables and
understand the structure of the datasets before preprocessing.

In [1]:
import pandas as pd

matches = pd.read_csv("matches.csv")
seasonstats = pd.read_csv("seasonstats.csv")

print(matches.columns.tolist())
print("\n")
print(seasonstats.columns.tolist())

['Unnamed: 0', 'Season', 'Date', 'Home', 'xG', 'Home Goals', 'Away Goals', 'xG.1', 'Away', 'Attendance', 'Venue']


['Unnamed: 0', 'Season', 'Squad', 'W', 'D', 'L', 'GF', 'GA', 'Pts', 'Sh', 'SoT', 'FK', 'PK', 'Cmp', 'Att', 'Cmp%', 'CK', 'CrdY', 'CrdR', 'Fls', 'PKcon', 'OG']


In [2]:
matches.head()

,Unnamed: 0,Season,Date,Home,xG,Home Goals,Away Goals,xG.1,Away,Attendance,Venue
0,0,2023/2024,2023-08-11,Burnley,0.3,0.0,3.0,1.9,Manchester City,21572.0,Turf Moor
1,1,2023/2024,2023-08-12,Arsenal,0.8,2.0,1.0,1.2,Nott'ham Forest,59984.0,Emirates Stadium
2,2,2023/2024,2023-08-12,Everton,2.7,0.0,1.0,1.5,Fulham,39940.0,Goodison Park
3,3,2023/2024,2023-08-12,Sheffield Utd,0.5,0.0,1.0,1.9,Crystal Palace,31194.0,Bramall Lane
4,4,2023/2024,2023-08-12,Brighton,4.0,4.0,1.0,1.5,Luton Town,31872.0,The American Express Community Stadium


In [3]:
matches.shape

(60529, 11)

In [4]:
matches.isnull().sum()

Unnamed: 0        0
Season            0
Date           9961
Home           9961
xG            57869
Home Goals     9961
Away Goals     9961
xG.1          57869
Away           9961
Attendance    49083
Venue         48123
dtype: int64

# Filtering Modern Premier League Seasons

The original dataset contains Premier League matches dating back to
the late 19th century. However, older seasons contain substantial
amounts of missing data and are less representative of modern football.

To ensure consistency and improve data quality, the analysis focuses
on seasons from 2013/2014 to 2023/2024. This period represents the
modern Premier League era and also covers the post-Sir Alex Ferguson
period at Manchester United, which is central to the project's narrative.

Restricting the dataset to modern seasons also improves the availability
of advanced metrics such as expected goals (xG).

In [5]:
# Filter the dataset to include only modern Premier League seasons between 2013/2014 and 2023/2024.
modern = matches[matches["Season"].isin([
    "2013/2014","2014/2015","2015/2016",
    "2016/2017","2017/2018","2018/2019",
    "2019/2020","2020/2021","2021/2022",
    "2022/2023","2023/2024"
])]

modern.shape

(4704, 11)

In [6]:
# Check the number of missing values in each column
modern.isnull().sum()

Unnamed: 0       0
Season           0
Date           524
Home           524
xG            2044
Home Goals     524
Away Goals     524
xG.1          2044
Away           524
Attendance     965
Venue          524
dtype: int64

In [7]:
# Remove matches with missing essential information such as team names or final scores. 
# These variables are fundamental for constructing the target variable and transforming the dataset into a team-centric format.
modern = modern.dropna(subset=[
    "Home",
    "Away",
    "Home Goals",
    "Away Goals"
])

In [8]:
modern.shape

(4180, 11)

In [9]:
# Create a dataframe representing the home team's perspective
# for each match.

home_df = pd.DataFrame({
    "Season": modern["Season"],
    "Date": modern["Date"],
    "Team": modern["Home"],
    "Opponent": modern["Away"],
    "Goals_For": modern["Home Goals"],
    "Goals_Against": modern["Away Goals"],
    "xG_For": modern["xG"],
    "xG_Against": modern["xG.1"],
    "Attendance": modern["Attendance"],

    # Indicator variable showing that the team played at home
    "Home": 1
})

# Create a dataframe representing the away team's perspective
# for each match.

away_df = pd.DataFrame({
    "Season": modern["Season"],
    "Date": modern["Date"],
    "Team": modern["Away"],
    "Opponent": modern["Home"],
    "Goals_For": modern["Away Goals"],
    "Goals_Against": modern["Home Goals"],
    "xG_For": modern["xG.1"],
    "xG_Against": modern["xG"],
    "Attendance": modern["Attendance"],

    # Indicator variable showing that the team played away
    "Home": 0
})

# Combine both perspectives into a single dataset
# where each row represents one team's performance in a match.

team_matches = pd.concat([home_df, away_df], ignore_index=True)

In [10]:
# Create the target variable for the classification task.
# A value of 1 indicates that the team won the match, while 0 represents either a draw or a loss.
team_matches["Win"] = (
    team_matches["Goals_For"] >
    team_matches["Goals_Against"]
).astype(int)

In [11]:
team_matches.head()
team_matches.shape

(8360, 11)

In [12]:
# Convert the Date column into a proper datetime format to allow chronological sorting and rolling calculations.

team_matches["Date"] = pd.to_datetime(team_matches["Date"])

# Sort matches by team and date so that rolling statistics are calculated in the correct chronological order.

team_matches = team_matches.sort_values(
    by=["Team", "Date"]
)

In [13]:
# Calculate goal difference for each match from the team's perspective.
# Positive values indicate stronger performances,
# while negative values indicate weaker performances.
team_matches["Goal_Diff"] = (
    team_matches["Goals_For"] -
    team_matches["Goals_Against"]
)

In [14]:
# Calculate expected goals (xG) difference for each match. This metric estimates the difference in chance quality
# between a team and its opponent and is often more informative than raw scorelines alone.
team_matches["xG_Diff"] = (
    team_matches["xG_For"] -
    team_matches["xG_Against"]
)

In [15]:
# Calculate the rolling average of goals scored over the previous 5 matches.
# The current match is excluded using shift(1) in order to prevent data leakage and ensure that only past information is used.
team_matches["Rolling_GF"] = (
    team_matches
    .groupby("Team")["Goals_For"]
    .transform(
        lambda x: x.shift(1).rolling(5).mean()
    )
)

In [16]:
# Calculate the rolling average of goals conceded over the previous 5 matches.
# This feature captures recent defensive performance while ensuring that only information from past matches is used.
team_matches["Rolling_GA"] = (
    team_matches
    .groupby("Team")["Goals_Against"]
    .transform(
        lambda x: x.shift(1).rolling(5).mean()
    )
)

In [17]:
# Calculate the rolling average of expected goals difference (xG_Diff) over the previous 5 matches.
# This feature represents recent chance creation and defensive control, providing a more stable measure of team performance than raw results alone.

team_matches["Rolling_xG_Diff"] = (
    team_matches
    .groupby("Team")["xG_Diff"]
    .transform(
        lambda x: x.shift(1).rolling(5).mean()
    )
)

In [18]:
# Calculate the rolling win rate over the previous 5 matches.
# This feature captures recent team form by measuring the proportion of matches won in the recent past.
team_matches["Rolling_Win_Rate"] = (
    team_matches
    .groupby("Team")["Win"]
    .transform(
        lambda x: x.shift(1).rolling(5).mean()
    )
)

In [19]:
# Check the number of missing values after feature engineering.
team_matches.isnull().sum()

Season                 0
Date                   0
Team                   0
Opponent               0
Goals_For              0
Goals_Against          0
xG_For              3040
xG_Against          3040
Attendance           882
Home                   0
Win                    0
Goal_Diff              0
xG_Diff             3040
Rolling_GF           170
Rolling_GA           170
Rolling_xG_Diff     3190
Rolling_Win_Rate     170
dtype: int64

In [20]:
# Create the full modeling dataset using features available for nearly all matches.

full_data = team_matches.dropna(subset=[
    "Rolling_GF",
    "Rolling_GA",
    "Rolling_Win_Rate"
])

In [21]:
# Create a second dataset containing only matches where expected goals (xG) data is available.

xg_data = team_matches.dropna(subset=[
    "xG_For",
    "xG_Against",
    "Rolling_xG_Diff"
])

In [22]:
print(full_data.shape)
print(xg_data.shape)

(8190, 17)
(5170, 17)


In [24]:
# Define the feature set for the full historical model. These features are based on information available before each match,
# such as recent attacking form, defensive form, home advantage,
# and recent win rate.

features_full = [
    "Home",
    "Rolling_GF",
    "Rolling_GA",
    "Rolling_Win_Rate",
    "Attendance"
]

# Define the feature set for the advanced xG-based model.
# In addition to the previous features, this model also includes
# rolling expected goals difference, which captures recent chance quality.

features_xg = [
    "Home",
    "Rolling_GF",
    "Rolling_GA",
    "Rolling_Win_Rate",
    "Attendance",
    "Rolling_xG_Diff"
]

# Define the target variable for the classification task.
# A value of 1 represents a win, while 0 represents
# either a draw or a loss.

target = "Win"

In [25]:
# Create the input matrix (X) and target vector (y)
# for the full historical model.

X_full = full_data[features_full]
y_full = full_data[target]

# Create the input matrix (X) and target vector (y)
# for the advanced xG-based model.

X_xg = xg_data[features_xg]
y_xg = xg_data[target]

In [26]:
# Split both datasets into training and testing subsets.

# The training data will be used to train the models,
# while the testing data will be used to evaluate
# how well the models generalize to unseen matches.

from sklearn.model_selection import train_test_split

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=42
)

X_train_xg, X_test_xg, y_train_xg, y_test_xg = train_test_split(
    X_xg,
    y_xg,
    test_size=0.2,
    random_state=42
)

In [35]:
# Standardize the numerical features so that variables
# with larger scales do not dominate the models.

# Scaling is particularly important for models such as
# Logistic Regression.

from sklearn.preprocessing import StandardScaler

scaler_full = StandardScaler()
X_train_full_scaled = scaler_full.fit_transform(X_train_full)
X_test_full_scaled = scaler_full.transform(X_test_full)

scaler_xg = StandardScaler()
X_train_xg_scaled = scaler_xg.fit_transform(X_train_xg)
X_test_xg_scaled = scaler_xg.transform(X_test_xg)

In [36]:
X_train_full.isnull().sum()

Home                0
Rolling_GF          0
Rolling_GA          0
Rolling_Win_Rate    0
Attendance          0
dtype: int64

In [37]:
# Replace missing attendance values using median imputation.

attendance_median = X_train_full["Attendance"].median()

X_train_full["Attendance"] = X_train_full["Attendance"].fillna(attendance_median)
X_test_full["Attendance"] = X_test_full["Attendance"].fillna(attendance_median)

In [38]:
# Import the Logistic Regression model
# and evaluation metrics.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Initialize the Logistic Regression model.

log_model = LogisticRegression(random_state=42)

# Train the model using the scaled training data.

log_model.fit(X_train_full_scaled, y_train_full)

# Generate predictions on the testing dataset.

y_pred_log = log_model.predict(X_test_full_scaled)

# Evaluate model performance using classification accuracy.

log_accuracy = accuracy_score(y_test_full, y_pred_log)

print("Logistic Regression Accuracy:", log_accuracy)

# Display a detailed classification report including
# precision, recall, and F1-score.

print(classification_report(y_test_full, y_pred_log))

Logistic Regression Accuracy: 0.6434676434676435
              precision    recall  f1-score   support

           0       0.66      0.87      0.75      1004
           1       0.58      0.28      0.38       634

    accuracy                           0.64      1638
   macro avg       0.62      0.58      0.56      1638
weighted avg       0.63      0.64      0.61      1638



## Logistic Regression Results

The Logistic Regression model achieved an accuracy of approximately 64%.
The model performed reasonably well at identifying matches that did not
result in a win, but struggled more when predicting victories.

This behavior suggests that recent form and home advantage contain useful
predictive information, but the relationship between football performance
and match outcomes is likely nonlinear and more complex than a linear model
can fully capture.

These results motivate the use of more advanced ensemble-based models
such as Random Forest and Gradient Boosting in later stages of the project.

In [40]:
# Import the Random Forest classifier and evaluation metrics.

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize the Random Forest model.

# Random Forest is an ensemble-based method that combines
# multiple decision trees in order to capture nonlinear
# relationships within the data.

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

# Train the model using the full training dataset.

rf_model.fit(X_train_full, y_train_full)

# Generate predictions on the testing dataset.

y_pred_rf = rf_model.predict(X_test_full)

# Evaluate model accuracy.

rf_accuracy = accuracy_score(y_test_full, y_pred_rf)

print("Random Forest Accuracy:", rf_accuracy)

# Display detailed classification metrics.

print(classification_report(y_test_full, y_pred_rf))

Random Forest Accuracy: 0.6428571428571429
              precision    recall  f1-score   support

           0       0.66      0.85      0.74      1004
           1       0.57      0.32      0.41       634

    accuracy                           0.64      1638
   macro avg       0.62      0.58      0.58      1638
weighted avg       0.63      0.64      0.61      1638



In [41]:
# Import the Gradient Boosting classifier and evaluation metrics.

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize the Gradient Boosting model.

# Gradient Boosting builds decision trees sequentially,
# where each new tree attempts to correct the errors
# made by previous trees.

gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

# Train the model using the training dataset.

gb_model.fit(X_train_full, y_train_full)

# Generate predictions on the testing dataset.

y_pred_gb = gb_model.predict(X_test_full)

# Evaluate classification accuracy.

gb_accuracy = accuracy_score(y_test_full, y_pred_gb)

print("Gradient Boosting Accuracy:", gb_accuracy)

# Display detailed classification metrics.

print(classification_report(y_test_full, y_pred_gb))

Gradient Boosting Accuracy: 0.645909645909646
              precision    recall  f1-score   support

           0       0.67      0.85      0.75      1004
           1       0.57      0.33      0.42       634

    accuracy                           0.65      1638
   macro avg       0.62      0.59      0.58      1638
weighted avg       0.63      0.65      0.62      1638

